# Historical research notebook

Sanitized source archive, not a validated current model. Saved outputs were removed. Data, credentials and model weights are not included. Path placeholders are not automatically expanded. See docs/CODE_CHANGELOG.md.


# MLB 종결 모델 — Result Tracker
**매일 predictor 실행 후 이 노트북으로 결과 누적**

- 일별/주별/월별 적중률 추적
- 성능 하락 자동 감지 (60% 미만 경고)

In [ ]:
import pandas as pd
import numpy as np
import os
from google.colab import drive
drive.mount("/content/drive")

MLB_PATH = "<DRIVE_ROOT>/MLB데이터"
TRACKER_PATH = f"{MLB_PATH}/result_tracker_2026.csv"

# 기존 트래커 로드 (없으면 새로 생성)
if os.path.exists(TRACKER_PATH):
    tracker = pd.read_csv(TRACKER_PATH)
    print(f"✅ 기존 트래커 로드: {len(tracker)}건")
else:
    tracker = pd.DataFrame(columns=[
        "date","gamePk","away_team","home_team",
        "win_prob","win_conf","win_prediction","win_selection",
        "run_total_prob","run_total_conf","run_total_prediction","run_total_baseline","run_total_selection",
        "home_score","away_score","home_result","total_runs",
        "win_correct","run_total_correct"
    ])
    print("📝 새 트래커 생성")

print(f"   기간: {tracker['date'].min() if len(tracker)>0 else '없음'} ~ {tracker['date'].max() if len(tracker)>0 else '없음'}")

## 1. 오늘 결과 추가

In [ ]:
# ============================================================
# 오늘 날짜 입력 (predictor 돌린 날짜)
# ============================================================
ADD_DATE = "2026-04-01"  # ← 바꾸기

# predictor 결과가 메모리에 있으면 사용, 없으면 재구성
# → predictor 노트북 실행 후 이 노트북 실행 권장

DATA_2026 = f"{MLB_PATH}/2026"
OL_PATH = f"{MLB_PATH}/MARKET_PRICE,LINEUP"
results = pd.read_csv(f"{DATA_2026}/mlb_2026_정규시즌_경기결과.csv")
st = pd.read_csv(f"{OL_PATH}/mlb_starters_2026.csv")
market_price = pd.read_csv(f"{OL_PATH}/mlb_market_price_2026.csv")

# 해당 날짜 경기
day_results = results[results["날짜"]==ADD_DATE]
day_st = st[st["date"]==ADD_DATE]
day_market_price = market_price[market_price["date"]==ADD_DATE]

if len(day_results) == 0:
    print(f"⚠️ {ADD_DATE} 경기결과 없음 — 경기결과 CSV 업데이트 필요")
else:
    # 이미 트래커에 있는 날짜면 스킵
    if len(tracker) > 0 and ADD_DATE in tracker["date"].values:
        existing = len(tracker[tracker["date"]==ADD_DATE])
        print(f"⚠️ {ADD_DATE}는 이미 {existing}건 등록됨 — 중복 추가 안 함")
        print(f"   덮어쓰려면: tracker = tracker[tracker['date']!='{ADD_DATE}']")
    else:
        # today CSV에서 예측값 가져오기
        today_path = f"{DATA_2026}/today_{ADD_DATE}.csv"
        if os.path.exists(today_path):
            today = pd.read_csv(today_path)
            print(f"  today CSV: {len(today)}경기")
        else:
            print(f"  ⚠️ today CSV 없음 — starters+market_price로 대체")
            today = day_st.merge(day_market_price[["date","home_team","home_WIN_LOSS","run_total_baseline","run_margin"]], 
                                 on=["date","home_team"], how="left")
        
        print(f"  {ADD_DATE} 경기결과: {len(day_results)}건")
        print(f"  추가 준비 완료 — 다음 셀에서 predictor 재실행 후 결과 저장")

## 2. 결과 저장 (predictor 실행 후)

In [ ]:
# predictor 노트북에서 base 변수가 있어야 함
# 없으면 today CSV에서 예측 결과 로드 시도

try:
    _ = base
    print(f"✅ base 변수 존재 — {len(base)}경기")
except:
    # today CSV에 win_prob/run_total_prob가 저장돼 있으면 그걸 사용
    today_path = f"{DATA_2026}/today_{ADD_DATE}.csv"
    pred_path = f"{DATA_2026}/predictions_{ADD_DATE}.csv"
    if os.path.exists(pred_path):
        base = pd.read_csv(pred_path)
        print(f"✅ predictions CSV 로드 — {len(base)}경기")
    else:
        print("⚠️ base 변수 없음 + predictions CSV 없음")
        print("   → predictor 노트북 먼저 실행하세요")
        base = None

if base is not None and "홈팀결과" in base.columns:
    new_rows = []
    for _, g in base.iterrows():
        if pd.isna(g.get("홈팀결과")): continue
        
        win_prediction = g["home_team"] if g["win_prob"]>=0.5 else g["away_team"]
        run_total_prediction = "UPPER" if g["run_total_prob"]>=0.5 else "LOWER"
        home_won = g["홈팀결과"] == "승"
        total = g["홈팀득점"] + g["원정팀득점"]
        actual_upper = total > g["run_total_baseline"]
        push = total == g["run_total_baseline"]
        
        win_correct = int((g["win_prob"]>=0.5) == home_won) if g["win_conf"]>=0.64 else np.nan
        run_total_correct = int((g["run_total_prob"]>=0.5) == actual_upper) if g["run_total_conf"]>=0.60 and not push else np.nan
        
        new_rows.append({
            "date": g["date"],
            "gamePk": g["gamePk"],
            "away_team": g["away_team"],
            "home_team": g["home_team"],
            "win_prob": round(g["win_prob"], 3),
            "win_conf": round(g["win_conf"], 3),
            "win_prediction": win_prediction,
            "win_selection": "SELECTION" if g["win_conf"]>=0.64 else "",
            "run_total_prob": round(g["run_total_prob"], 3),
            "run_total_conf": round(g["run_total_conf"], 3),
            "run_total_prediction": run_total_prediction,
            "run_total_baseline": g["run_total_baseline"],
            "run_total_selection": "SELECTION" if g["run_total_conf"]>=0.60 else "",
            "home_score": g["홈팀득점"],
            "away_score": g["원정팀득점"],
            "home_result": g["홈팀결과"],
            "total_runs": total,
            "win_correct": win_correct,
            "run_total_correct": run_total_correct,
        })
    
    if new_rows:
        new_df = pd.DataFrame(new_rows)
        # 중복 체크 (ADD_DATE 기준)
        if len(tracker) > 0 and ADD_DATE in tracker["date"].values:
            print(f"⚠️ {ADD_DATE} 이미 존재 — 스킵")
        else:
            tracker = pd.concat([tracker, new_df], ignore_index=True)
            tracker.to_csv(TRACKER_PATH, index=False)
            print(f"✅ {len(new_rows)}건 추가 → 총 {len(tracker)}건 저장")
    else:
        print("⚠️ 저장할 결과 없음")

## 3. 누적 성적표

In [ ]:
# 트래커 다시 로드 (최신)
if os.path.exists(TRACKER_PATH):
    tracker = pd.read_csv(TRACKER_PATH)

if len(tracker) == 0:
    print("데이터 없음")
else:
    tracker["date"] = pd.to_datetime(tracker["date"])
    
    # 승패 SELECTION만
    win_selections = tracker[tracker["win_selection"]=="SELECTION"].copy()
    win_valid = win_selections[win_selections["win_correct"].notna()]
    
    # 총득점 SELECTION만
    run_total_selections = tracker[tracker["run_total_selection"]=="SELECTION"].copy()
    run_total_valid = run_total_selections[run_total_selections["run_total_correct"].notna()]
    
    SEP = "="*60
    print(SEP)
    print(f"  2026 시즌 누적 성적표")
    print(f"  기간: {tracker['date'].min().strftime('%m/%d')} ~ {tracker['date'].max().strftime('%m/%d')}")
    print(SEP)
    
    if len(win_valid) > 0:
        wa = win_valid["win_correct"].mean() * 100
        status = "🟢 정상" if wa >= 65 else "🟡 주의" if wa >= 60 else "🔴 점검필요"
        print(f"\n  승패: {wa:.1f}% ({int(win_valid['win_correct'].sum())}/{len(win_valid)}) {status}")
    
    if len(run_total_valid) > 0:
        oa = run_total_valid["run_total_correct"].mean() * 100
        status = "🟢 정상" if oa >= 65 else "🟡 주의" if oa >= 60 else "🔴 점검필요"
        print(f"  총득점: {oa:.1f}% ({int(run_total_valid['run_total_correct'].sum())}/{len(run_total_valid)}) {status}")
    
    # 일별 추이
    print(f"\n  --- 일별 ---")
    for date, grp in tracker.groupby("date"):
        wg = grp[(grp["win_selection"]=="SELECTION") & (grp["win_correct"].notna())]
        og = grp[(grp["run_total_selection"]=="SELECTION") & (grp["run_total_correct"].notna())]
        w_str = f"{wg['win_correct'].mean()*100:.0f}%({len(wg)})" if len(wg)>0 else "-"
        o_str = f"{og['run_total_correct'].mean()*100:.0f}%({len(og)})" if len(og)>0 else "-"
        print(f"    {date.strftime('%m/%d')}: 승패 {w_str:>10s} | 총득점 {o_str:>10s}")
    
    # 최근 7일 vs 전체
    if len(win_valid) >= 20:
        last7 = tracker["date"].max() - pd.Timedelta(days=7)
        recent_w = win_valid[win_valid["date"]>=last7]
        if len(recent_w) >= 5:
            rw = recent_w["win_correct"].mean()*100
            print(f"\n  최근7일 승패: {rw:.1f}% ({len(recent_w)}경기)")
            if rw < 60:
                print(f"  🔴 최근 성능 하락 감지 — 모델 점검 권장")